# ProjectCerebro Week 2 ML Training on Google Colab

Use a T4 GPU runtime. Before running this notebook, add a Google Drive shortcut from **Shared with me** to:

`My Drive/projectcerebro/ProjectCerebro-Shared`

This notebook trains from the shared Drive `delta_lake/` folder and saves artifacts under the Colab repo checkout. Copy artifacts back to Drive before disconnecting.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Clone repo and switch branch

This requires `chiranjeev/ml-core-week2` to be pushed to GitHub first.

In [ ]:
%cd /content
!rm -rf projectcerebro
!git clone https://github.com/JANARDHANAREDDYMS/projectcerebro.git
%cd /content/projectcerebro
!git fetch origin
!git switch chiranjeev/ml-core-week2

## 3. Set and verify Drive data paths

In [ ]:
import os
from pathlib import Path

DATA_ROOT = "/content/drive/MyDrive/projectcerebro/ProjectCerebro-Shared/delta_lake"
BP8_30 = f"{DATA_ROOT}/epochs_mi_v1_ch5_sr128_bp8_30"
BP4_38 = f"{DATA_ROOT}/epochs_mi_v1_ch5_sr128_bp4_38"

os.environ["DATA_ROOT"] = DATA_ROOT
os.environ["BP8_30"] = BP8_30
os.environ["BP4_38"] = BP4_38

for path in [BP8_30, BP4_38]:
    print(path, "exists=", Path(path).exists(), "delta_log=", Path(path, "_delta_log").exists())

In [ ]:
!ls "$BP8_30"
!ls "$BP8_30/_delta_log" | head

## 4. Install dependencies without replacing Colab CUDA PyTorch

In [ ]:
!grep -v "^torch==" requirements.txt > requirements_colab.txt
!pip install -r requirements_colab.txt

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

## 5. Run unit tests

In [ ]:
!pytest tests -v

## 6. Smoke training

In [ ]:
!python -m ml_core.experiments.train_smoke \
  --filter bp8_30 \
  --delta-path "$BP8_30"

## 7. ShallowConvNet baseline

In [ ]:
!python -m ml_core.experiments.train_shallow_baseline \
  --filter bp8_30 \
  --delta-path "$BP8_30"

## 8. EEGNet PhysioNet pretrain

In [ ]:
!python -m ml_core.experiments.pretrain_eegnet_physionet \
  --filter bp8_30 \
  --delta-path "$BP8_30"

## 9. EEGNet BCI fine-tune

In [ ]:
!python -m ml_core.experiments.finetune_eegnet_bci \
  --filter bp8_30 \
  --delta-path "$BP8_30" \
  --pretrained artifacts/checkpoints/eegnet_pretrain_bp8_30/best.pt

## 10. Inspect and persist artifacts

In [ ]:
!find artifacts/checkpoints -maxdepth 3 -type f | sort
!find artifacts/mlruns -maxdepth 3 -type f | head -50

In [ ]:
!mkdir -p "/content/drive/MyDrive/projectcerebro/training_artifacts"
!cp -r artifacts/checkpoints artifacts/reports artifacts/mlruns "/content/drive/MyDrive/projectcerebro/training_artifacts/"
!find "/content/drive/MyDrive/projectcerebro/training_artifacts" -maxdepth 2 -type d | sort